### Ingestión del archivo "country.json"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

#### Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark

In [0]:
countries_schema = "countryId INT, countryIsoCode STRING, countryName STRING"

In [0]:
countries_df = spark.read \
    .schema(countries_schema) \
    .json(f"{bronze_folder_path}/{v_file_date}/country.json")

### Paso 2 - Eliminar columas no deseadas del DataFrame

In [0]:
from pyspark.sql.functions import col

In [0]:
countries_dropped_df = countries_df.drop(col("countryIsoCode"))

#### Paso 3 - Cambiar el nombre de las columnas y añadir "ingestion_date" y "enviroment"

In [0]:
from pyspark.sql.functions import lit

In [0]:
countries_final_df = add_ingestion_date(countries_dropped_df) \
    .withColumnRenamed("countryId", "country_id") \
    .withColumnRenamed("countryName", "country_name") \
    .withColumn("environment", lit(v_environment)) \
    .withColumn("file_date", lit(v_file_date))

#### Paso 4 - Escribir la salida en un formato "Delta"

In [0]:
countries_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.countries")

In [0]:
%sql
SELECT * FROM movie_silver.countries;

country_id,country_name,ingestion_date,environment,file_date
128,United Arab Emirates,2026-09-21T16:17:18.030775Z,production,2024-12-16
129,Afghanistan,2026-09-21T16:17:18.030775Z,production,2024-12-16
130,Angola,2026-09-21T16:17:18.030775Z,production,2024-12-16
131,Argentina,2026-09-21T16:17:18.030775Z,production,2024-12-16
132,Austria,2026-09-21T16:17:18.030775Z,production,2024-12-16
133,Australia,2026-09-21T16:17:18.030775Z,production,2024-12-16
134,Aruba,2026-09-21T16:17:18.030775Z,production,2024-12-16
135,Bosnia and Herzegovina,2026-09-21T16:17:18.030775Z,production,2024-12-16
136,Belgium,2026-09-21T16:17:18.030775Z,production,2024-12-16
137,Bulgaria,2026-09-21T16:17:18.030775Z,production,2024-12-16


In [0]:
dbutils.notebook.exit("Success")